# N131 · 概率基础与随机采样

**目标：** 用等概率推导而非直方图宣称无偏。

**先修：** N042, N031, N124。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 概率空间；条件概率；期望；Fisher–Yates；蓄水池；加权采样；拒绝采样。

**配套讲解来源：** [同名逐章意见](../../comment/131_randomized_sampling_probability.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章要解决的问题是：**写一个"随机"算法很容易，但"看起来随机"不等于"每个结果真的等概率"。** 比如洗牌、从流式数据里抽 k 个样本、按权重抽奖、用 1~7 的随机数造出 1~10 的随机数——这些操作都要能用数学推导证明无偏，而不是"跑一万次画个直方图差不多平"就宣称正确（“学习目标”部分 的原话是"用等概率推导而非直方图宣称无偏"）。

给你具体到数字的例子：

- 输入：数组 `[0, 1, 2]`，一个能给出 0~2 和 0~1 均匀随机数的随机源
- 输出：`fisher_yates` 的某个排列，比如 `[2, 0, 1]`
- 为什么它无偏？洗牌只做两次抽签：第一轮从 3 个位置均匀选一个（3 种），第二轮从剩下 2 个位置均匀选一个（2 种），共 3×2 = 6 条等概率轨迹，而 3 个元素恰好有 3! = 6 种排列——一对一，每种排列的概率恰好是 1/6。“运行示例”部分 的表格穷举了这 6 条轨迹，每个排列各出现 1 次。

再看 rand10 的例子：你手里只有 `rand7()`（均匀返回 1~7），要造出均匀的 1~10。直接 `(rand7()+rand7())%10` 之类的凑法必有偏，本章做法是拼出 49 种等概率组合、只用其中能被 10 整分的 40 种，输出 1~10 各恰好 4 种组合支撑——每个结果严格 4/40 = 1/10。

## 2. 基础知识：先读懂这些词

- **概率空间**：一次随机实验所有可能结果的集合，加上每个结果的概率。本章反复用的手法是"数轨迹"：把算法用到的随机数序列看成一条条等概率路径，只要每条输出结果对应的路径条数相同，输出就等概率。
- **条件概率**：在已知某件事发生的前提下另一件事的概率。蓄水池推导里"新元素第 i 步被选中"之后"旧元素存活到第 i 步"就用到了它。
- **期望（expectation）**：随机变量按概率加权的平均值。比如 rand10 平均要调用几次 rand7：每轮成功概率 40/49，平均尝试 49/40 轮、每轮花 2 次调用，期望就是 2×49/40 ≈ 2.45 次。
- **Fisher–Yates 洗牌**：从最后一个位置往前走，第 i 轮从下标 0..i 里均匀选一个 j，交换 a[i] 和 a[j]。要点是"只能在尚未固定的位置里选"，选过的位置永远不再碰。这样恰好产生 n! 条等概率轨迹，对应 n! 种排列。
- **蓄水池抽样（reservoir sampling）**：数据像水流一样经过、长度事先不知道、只允许看一遍，却要均匀抽出 k 个。前 k 个先装满"水池"；第 i 个元素（i ≥ k）到来时以 k/i 的概率替换池中一个随机位置。
- **加权采样**：每个元素被抽中的概率正比于权重 w[i]。实现套路是把权重化成"抽签券"：元素 i 拿到 w[i] 张连号券，抽一张均匀的票落在谁的区间就选谁。
- **前缀和 + 二分查找**：加权采样的落地技术。把权重累加成前缀和数组，抽个 0..total−1 的随机数 ticket，用 `bisect_right` 找它落在哪个区间，O(log n) 定位中奖者。
- **拒绝采样（rejection sampling）**：手里的随机源产生不了目标分布时，先造一个更大的等概率空间，把"切分不均"的尾部样本直接丢弃重抽。rand10 用 rand7 拼出 49 格空间、拒绝 40 以后 9 格就是这个思想。
- **Random 种子（seed）**：`Random(131)` 这类写法固定了随机数序列的起点，让"随机"变得可复现——测试同一 seed 两次运行结果必须一模一样。

## 3. 思路推导：从小例子开始

- **Step 1：换掉"看起来随机"，改用"数轨迹"。** 判断一个随机算法是否无偏，唯一靠谱的办法是列出所有等概率的随机数序列，数一数每种输出被多少条序列支持。
- **Step 2：洗牌——每轮只在"未定区"里选。** 第 i 轮（i 从 n−1 递减到 1）从 0..i 均匀选 j 并交换。第 1 轮 n 种选法、第 2 轮 n−1 种……总共 n! 条轨迹；又因为每条轨迹唯一决定一个排列（抽签路径可逆推），每个排列恰好 1/n!。
- **Step 3：蓄水池——用"留任概率"做归纳。** 第 i 个新元素以 k/i 概率进池（随机替换池中一个位置）；已在池里的元素则要"躲过替换"才能存活。假设它在第 i−1 步后还在池中（概率 k/(i−1)），第 i 步被顶掉的机会只有 k/i × 1/i × ... 这笔账留到第五节完整算——结论是新旧元素在第 i 步之后的在池概率都是 k/i，人人平等。
- **Step 4：加权——把概率问题变成"票落谁家"。** 权重 (0, 2, 0, 3) 变成 5 张连号票：1 号元素占票 0~1，3 号元素占票 2~4；抽 0~4 一张均匀票，二分查找落在谁的领地。
- **Step 5：rand10——先扩张再拒绝。** 两次 rand7 拼出 49 种等概率组合（编号 0~48）；10 除不尽 49，尾部 40~48 这 9 格直接丢弃重抽，剩下的 40 格每 4 格对应一个输出。

**手算演示一：洗牌 [0,1,2] 的全部 6 条轨迹**（这正是 “运行示例”部分 表格的生成过程，`FixedRng([a,b])` 把随机数固定成指定值，从而可以穷举）：

| 第一轮 j (i=2, 选 0..2) | 第二轮 j (i=1, 选 0..1) | 过程 | 最终排列 |
|--------------------------|--------------------------|------|----------|
| 0 | 0 | [2,1,0] → 换 0,1 位 | (1, 2, 0) |
| 0 | 1 | [2,1,0] → 换 1,1 位（不动） | (2, 1, 0) |
| 1 | 0 | [0,2,1] → 换 0,1 位 | (2, 0, 1) |
| 1 | 1 | [0,2,1] → 不动 | (0, 2, 1) |
| 2 | 0 | [0,1,2] → 换 0,1 位 | (1, 0, 2) |
| 2 | 1 | [0,1,2] → 不动 | (0, 1, 2) |

六条轨迹每条概率 1/6，产出的恰好是全部 6 种排列各一次——这就是"每个排列 1/3! = 1/6"的现场证据。

**手算演示二：加权采样 WeightedSampler([0, 2, 0, 3])。** 前缀和数组逐项累加：prefix = [0, 2, 2, 5]，total = 5，也就是五张票 0~4。逐张票二分查找（`bisect_right` 找"第一个大于 ticket 的前缀和位置"）：

| ticket | 落在区间 | 中奖下标 |
|--------|----------|----------|
| 0 | [0,2) | 1 |
| 1 | [0,2) | 1 |
| 2 | [2,5) | 3 |
| 3 | [2,5) | 3 |
| 4 | [2,5) | 3 |

权重 2 的 1 号拿 2 张票、权重 3 的 3 号拿 3 张票、权重 0 的 0 号和 2 号一张也没有——被抽中概率严格等于 2/5、3/5、0、0。

**手算演示三：rand10 消化序列 [7,7,1,1]。** 第一轮 a=7,b=7，组合编号 (7−1)×7+(7−1) = 48 ≥ 40，落进被拒绝的尾部，丢弃重抽；第二轮 a=1,b=1，编号 0 < 40，输出 0 % 10 + 1 = 1。这解释了测试里 `rand10` 消化四个随机数却只输出一个值。

## 4. 核心代码：fisher_yates

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def fisher_yates(a, rng):
    for i in range(len(a) - 1, 0, -1):
        j = rng.randrange(i + 1)
        a[i], a[j] = (a[j], a[i])
    return a
```

### 逐段读懂代码

“分段实现”部分 包含两个顶层函数、一个类和 rand10。

**函数一：`fisher_yates(a, rng)` —— 原地均匀洗牌。**

```python
def fisher_yates(a, rng):
    for i in range(len(a) - 1, 0, -1):
        j = rng.randrange(i + 1)
        a[i], a[j] = (a[j], a[i])
    return a
```

- `for i in range(len(a)-1,0,-1)`：i 从最后一个下标倒着走到 1（走到 1 为止，因为只剩一个位置时没得选，i=0 那轮可以省）。位置 i 每轮结束就被"封存"，以后再也不会被碰。
- `j=rng.randrange(i+1)`：在 0..i 里均匀取一个下标。注意范围是 i+1 而不是 len(a)——这就是"只在未定区里选"的实现；选成全范围正是练习 2 要你构造的偏差错误。
- `a[i],a[j]=a[j],a[i]`：Python 的元组交换，把选中的元素换到"封存位" i 上。
- `return a`：原地修改后把同一个列表返回，方便链式使用。

**函数二：`reservoir_sample(stream, k, rng)` —— 单遍流式抽样。**

```python
def reservoir_sample(stream, k, rng):
    if k < 0:
        raise ValueError('nonnegative reservoir size required')
    reservoir = []
    for i, x in enumerate(stream):
        if i < k:
            reservoir.append(x)
        elif k:
            j = rng.randrange(i + 1)
            if j < k:
                reservoir[j] = x
    return reservoir
```

- `if k<0: raise ValueError(...)`：挡住负的池容量。k 是"要抽几个"，负数没有意义。
- `if i<k: reservoir.append(x)`：前 k 个元素直接进池——池还没满，谈不上替换。
- `elif k:`：紧凑写法，等价于 `elif k != 0`。k=0 时这个分支永远不进，随机数一次都不抽，池保持空——既正确又省随机数（测试里 k=0 返回 `[]` 靠的就是它）。
- `j=rng.randrange(i+1)`：在 0..i 里均匀取数（此时已经流过 i+1 个元素，这是"以 k/i 概率保留"的实现载体）。
- `if j<k: reservoir[j]=x`：j 落在前 k 个位置（概率 k/(i+1)，注意 i 从 0 计数，第 i+1 个元素对应概率 k/(i+1) = k/i，i 为 1 起数的编号）时，新元素顶掉池中第 j 项；否则新元素直接流走。
- 整个函数对 stream 只做一次遍历，stream 可以是生成器——这是它和"先存下来再随机抽"的本质区别。

**类：`WeightedSampler` —— 前缀和 + 二分的加权抽样器。**

```python
class WeightedSampler:

    def __init__(self, weights, rng=None):
        if any((not isinstance(w, int) or w < 0 for w in weights)) or sum(weights) <= 0:
            raise ValueError('nonnegative integer weights with positive total required')
        self.prefix = []
        total = 0
        for w in weights:
            total += w
            self.prefix.append(total)
        self.total = total
        self.rng = Random() if rng is None else rng

    def pick_index(self):
        ticket = self.rng.randrange(self.total)
        return bisect_right(self.prefix, ticket)
```

- `__init__` 的那行长校验：`any(not isinstance(w,int) or w<0 for w in weights)` 逐个检查"必须是整数且非负"（实现建立在"整数张票"上，浮点权重会破坏 randrange 的用法），`sum(weights)<=0` 再拦住"全零权重"（total=0 会让 randrange(0) 报错，且抽签无从谈起）。
- `for w in weights: total+=w; self.prefix.append(total)`：一遍累加出前缀和。权重 (0,2,0,3) 得 prefix=[0,2,2,5]，每个元素占据"上一个人的累计值到自己的累计值"这段票号区间。
- `self.rng=Random() if rng is None else rng`：三目表达式——调用方不给 rng 就自己新建一个 `Random()`；给了（比如测试里的 FixedRng）就用调用方的，这让随机行为可注入、可复现。
- `pick_index` 两行：`randrange(self.total)` 抽一张 0..total−1 的票；`bisect_right(self.prefix,ticket)` 二分找到第一个大于 ticket 的前缀和所在下标，即票落到的元素。权重为 0 的元素前后前缀和相同、区间长度为零，永远不会被抽中。

**函数三：`rand10(rand7)` —— 用 rand7 造 rand10（拒绝采样）。**

```python
def rand10(rand7):
    while True:
        a, b = rand7()
        if not 1 <= a <= 7 or not 1 <= b <= 7:
            raise ValueError('rand7 must return 1..7')
        ticket = (a - 1) * 7 + (b - 1)
        if ticket < 40:
            return ticket % 10 + 1
```

- `while True:`：一直重抽直到成功。拒绝采样天然是循环结构，不成功便成仁（每轮成功率 40/49，期望 2.45 次调用，几乎立刻成功）。
- `if not 1<=a<=7 or not 1<=b<=7: raise ValueError(...)`：校验传入的 rand7 真的返回 1..7。这是"合约检查"——上游违约时立刻报错，而不是让错误数据悄悄污染输出。
- `ticket=(a-1)*7+(b-1)`：把两次 rand7 拼成 0..48 的均匀组合编号。你可以想象一张 7×7 的表格，ticket 是格子的统一编号，49 格每格恰好 1/49。
- `if ticket<40: return ticket%10+1`：只取前 40 格，尾部 41~49 号（编号 40~48）的 9 格丢弃。40 = 4×10，每 10 格一组轮完 1~10，所以每个输出恰好 4 格支撑，概率 4/49 ÷ 40/49 = 1/10，严格均匀。

## 5. 分段实现：按顺序运行

**本章接口：** `fisher_yates(a, rng)`、`reservoir_sample(stream, k, rng)`、`WeightedSampler`、`rand10(rand7)`

### 导入本章使用的库

In [1]:
from bisect import bisect_right
from random import Random

### fisher_yates

In [2]:
def fisher_yates(a, rng):
    for i in range(len(a) - 1, 0, -1):
        j = rng.randrange(i + 1)
        a[i], a[j] = (a[j], a[i])
    return a

### reservoir_sample

In [3]:
def reservoir_sample(stream, k, rng):
    if k < 0:
        raise ValueError('nonnegative reservoir size required')
    reservoir = []
    for i, x in enumerate(stream):
        if i < k:
            reservoir.append(x)
        elif k:
            j = rng.randrange(i + 1)
            if j < k:
                reservoir[j] = x
    return reservoir

### WeightedSampler

In [4]:
class WeightedSampler:

    def __init__(self, weights, rng=None):
        if any((not isinstance(w, int) or w < 0 for w in weights)) or sum(weights) <= 0:
            raise ValueError('nonnegative integer weights with positive total required')
        self.prefix = []
        total = 0
        for w in weights:
            total += w
            self.prefix.append(total)
        self.total = total
        self.rng = Random() if rng is None else rng

    def pick_index(self):
        ticket = self.rng.randrange(self.total)
        return bisect_right(self.prefix, ticket)

### rand10

In [5]:
def rand10(rand7):
    while True:
        a, b = (rand7(), rand7())
        if not 1 <= a <= 7 or not 1 <= b <= 7:
            raise ValueError('rand7 must return 1..7')
        ticket = (a - 1) * 7 + (b - 1)
        if ticket < 40:
            return ticket % 10 + 1

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [6]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

from itertools import product
from collections import Counter
class FixedRng:
    def __init__(self,values): self.values=iter(values)
    def randrange(self,stop):
        value=next(self.values); assert 0<=value<stop; return value
counts=Counter(tuple(fisher_yates([0,1,2],FixedRng([a,b]))) for a,b in product(range(3),range(2)))
show_table(['排列','六条等概率抽签轨迹中的次数'],sorted(counts.items()))

排列,六条等概率抽签轨迹中的次数
"(0, 1, 2)",1
"(0, 2, 1)",1
"(1, 0, 2)",1
"(1, 2, 0)",1
"(2, 0, 1)",1
"(2, 1, 0)",1


## 7. 正确性、适用条件与复杂度

洗牌每个最终排列对应唯一抽签路径，概率为1/n!。蓄水池中旧元素在第i步后存活概率(k/(i−1))·(1−1/i)=k/i，新元素被保留概率也是k/i；完整均匀k子集性质可继续按子集归纳。加权区间包含恰好w[i]张票；拒绝抽样接受的40张票对10个结果各贡献4张。

**代价：** 洗牌O(n)时间O(1)辅助；蓄水池O(n)时间O(k)空间且仅遍历一次；加权初始化O(n)、每次O(log n)；rand10期望调用rand7次数2·49/40，无确定有限最坏次数。统计频率仅辅助诊断，测试主要使用精确轨迹枚举。

### 展开理解

**洗牌为什么均匀？** 数轨迹：第 1 轮 n 种选法、第 2 轮 n−1 种、……、最后一轮 2 种，总共 n! 条随机数序列，每条概率 1/n!。关键的第二步是"一条轨迹唯一决定一个排列"：知道最终排列和"每轮 i 交换到 i 位"的规则，可以倒推出每轮 j 的取值。n! 条轨迹一对一铺满 n! 种排列，每种排列概率 1/n!。第三节 [0,1,2] 的六行表格就是 n=3 的完整现场。

**蓄水池里新旧元素为什么概率相同？** 看第 i 个元素到来这一步（i ≥ k，编号从 1 数）。新元素被保留下来的条件是"随机数 j 落在 0..k−1"，概率 k/i。旧元素分两步：它先要在前 i−1 步后仍在池中（概率 k/(i−1)，当作已知），再在第 i 步不被顶掉。新元素若进池，会均匀挑池中一个位置替换，挑中它那个位置的机会是 (k/i)×(1/k) = 1/i，所以存活概率是 k/(i−1) × (1 − 1/i) = k/(i−1) × (i−1)/i = k/i。新旧都是 k/i，每一步之后"每个已看过的元素在池里的概率相等"，一路推到流结束，抽到的就是均匀的 k 元子集（“正确性与复杂度”部分 提到完整的"k 子集均匀性"可以对子集规模继续同样的归纳）。

**加权采样为什么正比于权重？** 元素 i 的领地长度恰好是 w[i] 张票（前缀和的差），总票数是所有权重的和，抽中概率 = w[i]/total，严格正比。零权重领地为零，永远抽不中。

**rand10 为什么严格 1/10？** 49 格等概率（每格 1/49），接受其中的 40 格；40 格按编号 % 10 分成 10 组、每组恰好 4 格。于是每个输出 = 4 格 × 1/49 ÷ (40/49) = 1/10。

**复杂度（结合具体数字）：** 洗牌 O(n) 时间、O(1) 额外空间（原地交换）——n = 10⁶ 也只是百万次交换，一秒内跑完。蓄水池 O(n) 时间、O(k) 空间、只遍历一遍——n = 10⁹ 的流只花 10⁹ 次循环和 k 个格子内存。加权采样初始化 O(n) 建前缀和，之后每次抽取只要一次 randrange 加一次二分，O(log n)——100 万元素的表抽取一次也就 20 次比较。rand10 期望调用 2×49/40 ≈ 2.45 次 rand7；要记住它**没有确定的最坏次数**（“正确性与复杂度”部分 明确标注），运气差时理论上会连续拒绝，只是概率衰减得极快。另外 “正确性与复杂度”部分 特别说明：统计频率只用于辅助诊断，真正的测试靠 FixedRng 穷举精确轨迹。

## 8. 单元测试：每个用例在检查什么

1. 洗牌穷举：把 6 条 (a,b) 组合全部喂给 `FixedRng`，统计 `fisher_yates([0,1,2])` 的输出。`assert len(counts)==6 and set(counts.values())=={1}` 要求恰好 6 种排列、每种 1 次——"集合去重后只剩 {1}"这种写法同时检查了"不重"和"不漏"。
2. 蓄水池穷举：流 range(4)、k=2，随机数序列是 (a∈0..2, b∈0..3) 共 12 条轨迹，对每条结果排序后计数。要求 6 种二元子集各出现 2 次——12 条轨迹均分给 6 个子集，均匀性落到实处。
3. `assert [WeightedSampler([0,2,0,3],FixedRng([i])).pick_index() for i in range(5)]==[1,1,3,3,3]` —— 把 5 张票逐张人工"抽出"，核对中奖下标序列：票 0、1 归 1 号，票 2、3、4 归 3 号；权重 0 的 0 号、2 号从头到尾不见踪影。这正是第三节演示二的表格直接转成断言。
4. `assert reservoir_sample(range(10),3,Random(131))==reservoir_sample(range(10),3,Random(131))` —— **可复现性测试**：同一种子两次运行必须逐位相同，验证随机源注入没有暗藏全局状态。
5. `assert reservoir_sample(iter([1,2]),5,Random(1))==[1,2]` —— **边界值**：流比池还短时，所有元素都进池、一次替换都不发生。
6. `assert reservoir_sample(range(5),0,Random(1))==[]` —— **边界值**：k=0 应返回空列表，且（由 `elif k:`）连随机数都不该消费。
7. rand10 全量验收：穷举 49 对 (a,b)，只统计 ticket<40 被接受的 40 对，`accepted==Counter({i:4 for i in range(1,11)})` 要求 1~10 每个输出恰好 4 条轨迹——均匀性的最强形式检查。
8. `values=iter([7,7,1,1]); assert rand10(...)==1` —— **拒绝路径测试**：先喂一对 (7,7)（编号 48 被拒），再喂 (1,1)（编号 0）输出 1，验证重抽循环真的工作。

In [7]:
from itertools import product

from collections import Counter

class FixedRng:

    def __init__(self, values):
        self.values = iter(values)

    def randrange(self, stop):
        value = next(self.values)
        assert 0 <= value < stop
        return value

counts = Counter((tuple(fisher_yates([0, 1, 2], FixedRng([a, b]))) for a, b in product(range(3), range(2))))

assert len(counts) == 6 and set(counts.values()) == {1}

subsets = Counter((tuple(sorted(reservoir_sample(range(4), 2, FixedRng([a, b])))) for a, b in product(range(3), range(4))))

assert len(subsets) == 6 and set(subsets.values()) == {2}

assert [WeightedSampler([0, 2, 0, 3], FixedRng([i])).pick_index() for i in range(5)] == [1, 1, 3, 3, 3]

assert reservoir_sample(range(10), 3, Random(131)) == reservoir_sample(range(10), 3, Random(131))

assert reservoir_sample(iter([1, 2]), 5, Random(1)) == [1, 2]

assert reservoir_sample(range(5), 0, Random(1)) == []

accepted = Counter()

for a, b in product(range(1, 8), repeat=2):
    if (a - 1) * 7 + b - 1 < 40:
        values = iter([a, b])
        accepted[rand10(lambda: next(values))] += 1

assert accepted == Counter({i: 4 for i in range(1, 11)})

values = iter([7, 7, 1, 1])

assert rand10(lambda: next(values)) == 1

print('N131: 所有本章断言通过')

N131: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 证明单元素蓄水池每项概率1/n。

**练习 2：** 构造错误全范围交换shuffle的偏差。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1（证明单元素蓄水池每项概率 1/n）：** 提示——k=1 时池里只有一个位置。第 i 个元素进池的概率是 1/i；它要活到最后，必须躲过第 i+1 到第 n 步每一步的替换，第 t 步被替换的概率是 (1/t)×(1/1) = 1/t，存活是 ∏(1 − 1/t)（t 从 i+1 到 n）。把这个连乘 telescope 展开：∏ (t−1)/t = i/n，再乘上 1/i 恰好 1/n。手算 n=3、i=2 验证：1/2 × (1−1/3) = 1/2 × 2/3 = 1/3。最后用 `reservoir_sample` 配 FixedRng 穷举小 n 的全部轨迹对账。

**练习 2（构造错误洗牌的偏差）：** 提示——把 `j=rng.randrange(i+1)` 改成 `j=rng.randrange(len(a))`（每轮都全范围选），表面看"更随机"，其实轨迹数变成 nⁿ 条而排列只有 n! 种，nⁿ 除不尽 n! 时必有偏。拿 n=3 动手数：27 条轨迹摊给 6 种排列，你会在计数表里看到某个排列出现 4 次、另一个 5 次（总 27），"set(counts.values())=={1}" 这类断言立刻失败。用本章的 FixedRng 穷举法生成这张 27 行计数表，就是最有说服力的反例证据。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [8]:
# 完整实现：本单元格不依赖其他单元格。
from bisect import bisect_right

from random import Random

def fisher_yates(a, rng):
    for i in range(len(a) - 1, 0, -1):
        j = rng.randrange(i + 1)
        a[i], a[j] = (a[j], a[i])
    return a

def reservoir_sample(stream, k, rng):
    if k < 0:
        raise ValueError('nonnegative reservoir size required')
    reservoir = []
    for i, x in enumerate(stream):
        if i < k:
            reservoir.append(x)
        elif k:
            j = rng.randrange(i + 1)
            if j < k:
                reservoir[j] = x
    return reservoir

class WeightedSampler:

    def __init__(self, weights, rng=None):
        if any((not isinstance(w, int) or w < 0 for w in weights)) or sum(weights) <= 0:
            raise ValueError('nonnegative integer weights with positive total required')
        self.prefix = []
        total = 0
        for w in weights:
            total += w
            self.prefix.append(total)
        self.total = total
        self.rng = Random() if rng is None else rng

    def pick_index(self):
        ticket = self.rng.randrange(self.total)
        return bisect_right(self.prefix, ticket)

def rand10(rand7):
    while True:
        a, b = (rand7(), rand7())
        if not 1 <= a <= 7 or not 1 <= b <= 7:
            raise ValueError('rand7 must return 1..7')
        ticket = (a - 1) * 7 + (b - 1)
        if ticket < 40:
            return ticket % 10 + 1

# 示例与回归测试
from itertools import product

from collections import Counter

class FixedRng:

    def __init__(self, values):
        self.values = iter(values)

    def randrange(self, stop):
        value = next(self.values)
        assert 0 <= value < stop
        return value

counts = Counter((tuple(fisher_yates([0, 1, 2], FixedRng([a, b]))) for a, b in product(range(3), range(2))))

assert len(counts) == 6 and set(counts.values()) == {1}

subsets = Counter((tuple(sorted(reservoir_sample(range(4), 2, FixedRng([a, b])))) for a, b in product(range(3), range(4))))

assert len(subsets) == 6 and set(subsets.values()) == {2}

assert [WeightedSampler([0, 2, 0, 3], FixedRng([i])).pick_index() for i in range(5)] == [1, 1, 3, 3, 3]

assert reservoir_sample(range(10), 3, Random(131)) == reservoir_sample(range(10), 3, Random(131))

assert reservoir_sample(iter([1, 2]), 5, Random(1)) == [1, 2]

assert reservoir_sample(range(5), 0, Random(1)) == []

accepted = Counter()

for a, b in product(range(1, 8), repeat=2):
    if (a - 1) * 7 + b - 1 < 40:
        values = iter([a, b])
        accepted[rand10(lambda: next(values))] += 1

assert accepted == Counter({i: 4 for i in range(1, 11)})

values = iter([7, 7, 1, 1])

assert rand10(lambda: next(values)) == 1

print('N131: 所有本章断言通过')

N131: 所有本章断言通过


## 11. 代表题与迁移

- **384. Shuffle an Array（打乱数组）**：每次调用返回等概率排列——`fisher_yates` 的原型题，练"倒序 + 未定区内选 j"的标准洗牌。
- **398. Random Pick Index（随机数索引）**：从重复数字的所有下标里均匀挑一个——单元素蓄水池（k=1）的直接应用，正是练习 1 证明的那个 1/n 场景。
- **528. Random Pick with Weight（按权重随机选择）**：按权重抽下标——`WeightedSampler` 的原型题，练前缀和 + 二分定位票落谁家。
- **470. Implement Rand10() Using Rand7()（用 Rand7 实现 Rand10）**：拒绝采样的教科书题——两次 rand7 拼 49 格、留 40 弃 9，`rand10` 的原样出处。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。